# Aurora in Jupyter

This notebook walks through using Aurora's Python SDK inside a Jupyter / JupyterLab / VS Code notebook environment. Everything Aurora can do — running an analysis, filtering findings, saving + verifying a bundle, exporting a shareable artifact — works inline.

**Prerequisites:**

```bash
pip install -r requirements.txt   # Aurora's runtime stack
pip install jupyterlab            # if you don't have it yet
```

Aurora runs entirely on your machine. No cloud calls, no API keys. The LLM that writes synthesis narratives runs locally too (default: Gemma 3 via Ollama).

## 1. Hello, Aurora

Drop a DataFrame in. Get findings out.

In [ ]:
import pandas as pd
import aurora_sdk as aurora

# Any DataFrame works. We'll grab NVDA daily prices for a recognisable demo.
# In production you'd use yfinance, your DB, an API call — anything that
# returns a DataFrame.
df = pd.read_csv('../../data/uploads/demo/nvda_clean.csv')
df.head()

In [ ]:
# Hand the DataFrame to Aurora. Quick depth ~ 14 seconds on consumer hardware.
# Aurora writes the DataFrame to a CSV in outputs/ so the audit trail is preserved
# (Aurora hashes the CSV, not the in-memory df).
r = aurora.run(df, depth='quick', dataset_name='nvda_jupyter_demo')

# Just evaluate the result: Aurora renders a summary card automatically.
r

## 2. Explore the findings

`r.findings` is a chainable filter view. Evaluate it directly for a sortable-looking table.

In [ ]:
# All findings as an HTML table.
r.findings

In [ ]:
# Only critical findings.
r.findings.critical()

In [ ]:
# Filter by method — useful when you want to dig into one specific analytical result.
r.findings.by_method('hmm_baum_welch')

In [ ]:
# Drop to raw dicts whenever you want to compose with anything else.
import pandas as pd
findings_df = pd.DataFrame(r.findings.to_list())
findings_df[['severity', 'confidence', 'method', 'title']].head(10)

## 3. The bundle: Aurora's reproducible artifact

Every Aurora run produces a **Bundle** — a versioned, hashable, optionally-signable JSON document. The bundle is the unit of reproducibility: anyone with the same Aurora version + the same bundle can verify every claim.

In [ ]:
r.bundle

In [ ]:
# Save it — single JSON file, portable across machines.
r.bundle.save('nvda_analysis.aurora.json')

# Re-load it (e.g., on a reviewer's machine).
b = aurora.Bundle.load('nvda_analysis.aurora.json')

# Integrity check: SHA-256 of the canonical JSON serialisation.
# If anyone changed even one z-score, this returns False.
print('verify() =', b.verify())

## 4. Export the notebook + bundle as one shareable artifact

When you want to send a teammate the **whole thing** (your narrative + the analytical evidence), use `export_notebook`. It produces a single `.aurora-notebook.tar.gz` containing this notebook, the bundle, and a manifest with integrity hashes.

In [ ]:
info = aurora.export_notebook(
    notebook_path='aurora_in_jupyter.ipynb',
    bundle=r.bundle,
    output_path='nvda_analysis_report',
)

print(f'Wrote: {info.output_path}')
print(f'Size:  {info.size_bytes // 1024} KB')
print(f'Notebook SHA-256: {info.notebook_sha256[:16]}...')
print(f'Bundle SHA-256:   {info.bundle_sha256[:16]}...')
print(f'Bundle content_hash: {info.bundle_content_hash[:16]}...')

## What just happened

- You handed Aurora a pandas DataFrame. Aurora wrote it to a CSV (preserved alongside the run_dir for audit) and ran 8 statistical methods on it.
- Aurora returned a `RunResult`. Evaluating it in a notebook rendered a summary card.
- `r.findings` produced a chainable filter view; evaluating it rendered an HTML table.
- `r.bundle.save(...)` wrote a portable, hashable JSON artifact. `Bundle.load(...).verify()` confirmed the math hasn't been tampered with.
- `aurora.export_notebook(...)` bundled this notebook + the analytical evidence into a single shareable tar — reviewer-ready.

**Every number above traces to a named statistical method. Every method cites a paper. `fabricated_count = 0` is the contractual signal.** 

Cloud LLMs guess. Aurora computes. 🐻